# 🧬 Voice Clone Studio MVP — Qwen3-TTS
A complete voice cloning workbench with options to use a URL sample, upload a file, or record from your microphone.

**Tips for good cloning:**
- Use 3-15 seconds of clean audio.
- Ensure an accurate transcript.
- No background noise for the best results.


In [ ]:
!pip install -q qwen-tts soundfile gradio

import os
import torch
import soundfile as sf
import gradio as gr
import urllib.request
from IPython.display import Audio, display, clear_output


In [ ]:
MODEL_SIZE = '1.7B'
MODEL_ID = f'Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-Base'
print(f'Using Model: {MODEL_ID}')


In [ ]:
from qwen_tts.model import Qwen3TTSModel

def to_wav(result, default_sr=24000):
    """Normalize any Qwen3-TTS generate_* return into (waveform, sample_rate)."""
    audio, sr = result if isinstance(result, tuple) else (result, default_sr)
    if isinstance(audio, (list, tuple)):
        audio = audio[0]
    if hasattr(audio, "cpu"):
        audio = audio.cpu().numpy()
    return audio, sr

print(f'Loading model {MODEL_ID}...')
base_model = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map='cuda:0',
    dtype=torch.bfloat16,
    attn_implementation='sdpa'
)
print('Model loaded successfully!')


## Set up Reference Audio
Below are three options. Option A is active by default. Uncomment B or C to try other methods.


In [ ]:
# Option A: Default URL (Official Qwen demo)
REF_AUDIO = 'clone.wav'
REF_TEXT = "It's actually kind of magical. Like, you can just talk, and it sounds like this. Crazy, right?"
urllib.request.urlretrieve('https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-TTS-Repo/clone.wav', REF_AUDIO)

# Option B: Upload via Google Colab
# from google.colab import files
# uploaded = files.upload()
# REF_AUDIO = list(uploaded.keys())[0]
# REF_TEXT = "ENTER_YOUR_TRANSCRIPT_HERE"

# Option C: Any URL
# REF_AUDIO = 'custom.wav'
# REF_TEXT = "ENTER_YOUR_TRANSCRIPT_HERE"
# urllib.request.urlretrieve('YOUR_URL_HERE', REF_AUDIO)

print(f'Reference Audio: {REF_AUDIO}')
print(f'Reference Text: {REF_TEXT}')
display(Audio(REF_AUDIO))


In [ ]:
print('Building voice clone prompt...')
voice_prompt = base_model.create_voice_clone_prompt(ref_audio=REF_AUDIO, ref_text=REF_TEXT)
print('Voice clone prompt created successfully!')


In [ ]:
test_sentences = [
    'Hello there! This is a test of the voice cloning system.',
    'I can say completely different things in your voice.',
    'Artificial intelligence is advancing at a rapid pace.',
    'How does this sound to you? Are the intonations natural?',
    'Thank you for trying out the Qwen3-TTS voice clone studio.'
]

for i, text in enumerate(test_sentences):
    print(f'Generating: {text}')
    audio = base_model.generate_voice_clone(text=text, language='English', voice_clone_prompt=voice_prompt)
    filename = f'clone_test_{i+1}.wav'
    audio, sr = to_wav(audio)
    sf.write(filename, audio, sr)
    display(Audio(filename))


In [ ]:
def generate_gradio(text):
    try:
        audio = base_model.generate_voice_clone(text=text, language='English', voice_clone_prompt=voice_prompt)
        filename = 'gradio_out.wav'
        audio, sr = to_wav(audio)
        sf.write(filename, audio, sr)
        return filename
    except Exception as e:
        print(f'Error: {e}')
        return None

iface = gr.Interface(
    fn=generate_gradio,
    inputs=gr.Textbox(lines=3, placeholder='Enter text to clone...', label='Text to Synthesize'),
    outputs=gr.Audio(label='Cloned Voice Output'),
    title='Voice Clone Studio',
    description='Generate speech using the pre-built cloned voice prompt.',
)
iface.launch(share=True, debug=True)


In [ ]:
!zip -q voice_clones.zip clone_test_*.wav
from google.colab import files
try:
    files.download('voice_clones.zip')
except:
    print('Download works only in Colab environment.')
